# SCRUM-24 — Leakage Audit & Method 2 Data Isolation

**Owner:** Mounik Sai · **Sprint:** Sprint 24 (Review 2 Preparation)

**Jira task:** *Leakage Audit & Method 2 Data Isolation - Isolate sentiment and topic variables from the dataset for Method 2 modeling. Verify that no future data leaks into lagged sentiment or topic trends.*

---

### Overview & Objectives
Method 1 (SCRUM-22) established a numerical time-series baseline utilizing Google Trends search interest and YouTube activity counts. Method 2 introduces consumer sentiment signals and topic distributions extracted from the SCRUM-23 YouTube text corpus (`data/processed/text_corpus.csv`, 29,189 comments across 60 products).

This interactive notebook demonstrates the end-to-end data isolation and leakage verification pipeline:
1. **Timezone Harmonization:** UTC timestamps localized to India Standard Time (`Asia/Kolkata`).
2. **Train-Only NLP Model Fitting:** LDA topic models and vectorizers fitted exclusively on training comments (`date < 2026-08-12`).
3. **Leak-Free Lag & Rolling Feature Engineering:** Strict backwards-looking shifts (`shift(lag)` and `shift(1).rolling(7)`).
4. **Quarantine Matrix:** Contemporaneous signals and point-in-time October 4 snapshot columns excluded from features.
5. **Automated 12-Point Leakage Audit:** Machine verification of zero lookahead bias and partition integrity.
6. **Chronological Train/Test Partitioning:** Strict 80/20 chronological partition (Cutoff: `2026-08-12`).

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import config
import isolate_method2_data as m2

paths = config.Paths.from_root(ROOT)
print(f"Project root: {ROOT}")
print(f"Master dataset: {paths.master_dataset_csv}")
print(f"Text corpus:    {paths.text_corpus_csv}")

Project root: /Users/mouniksai/Documents/viral-product-intelligence-
Master dataset: /Users/mouniksai/Documents/viral-product-intelligence-/processed/master_dataset.csv
Text corpus:    /Users/mouniksai/Documents/viral-product-intelligence-/data/processed/text_corpus.csv


## Step 1: Load Canonical Datasets (SCRUM-21 Master & SCRUM-23 Text Corpus)

In [2]:
df_master = pd.read_csv(paths.master_dataset_csv)
df_corpus = pd.read_csv(paths.text_corpus_csv)

print(f"Master Dataset: {df_master.shape[0]:,} rows x {df_master.shape[1]} columns")
print(f"Text Corpus:    {df_corpus.shape[0]:,} rows x {df_corpus.shape[1]} columns")
usable_comments = df_corpus[df_corpus['is_usable'].astype(str).str.lower().isin(['true', '1'])]
print(f"Usable comments: {len(usable_comments):,} ({len(usable_comments)/len(df_corpus)*100:.1f}%)")

Master Dataset: 16,140 rows x 43 columns
Text Corpus:    29,189 rows x 19 columns
Usable comments: 22,568 (77.3%)


## Step 2: Timezone Localization & Date Mapping Integrity (Asia/Kolkata)
YouTube comments record publish timestamps in UTC. To match Google Trends India calendar days, timestamps are converted to `Asia/Kolkata` (UTC+5:30).

In [3]:
dt_utc = pd.to_datetime(usable_comments['published_at'], utc=True)
dt_ist = dt_utc.dt.tz_convert('Asia/Kolkata')
usable_comments = usable_comments.copy()
usable_comments['date_ist'] = dt_ist.dt.strftime('%Y-%m-%d')

print(f"Sample UTC Timestamp: {usable_comments['published_at'].iloc[0]}")
print(f"Localized IST Date:   {usable_comments['date_ist'].iloc[0]}")
print(f"Date Range in IST:    {usable_comments['date_ist'].min()} to {usable_comments['date_ist'].max()}")

Sample UTC Timestamp: 2026-09-16T16:12:59Z
Localized IST Date:   2026-09-16
Date Range in IST:    2023-09-21 to 2026-10-04


## Step 3: Sentiment Scoring on `text_sentiment_ready`
Compound polarity scores are computed on the sentiment-cleaned text, preserving negations, punctuation, and emojis.

In [4]:
sent_df = m2.score_comment_sentiment(usable_comments['text_sentiment_ready'])
usable_comments['sentiment_compound'] = sent_df['compound'].values
usable_comments['is_pos'] = sent_df['is_pos'].values
usable_comments['is_neg'] = sent_df['is_neg'].values

print(f"Mean Sentiment Polarity: {usable_comments['sentiment_compound'].mean():+.4f}")
print(f"Positive Comments Share: {usable_comments['is_pos'].mean()*100:.1f}%")
print(f"Negative Comments Share: {usable_comments['is_neg'].mean()*100:.1f}%")
print(f"Neutral Comments Share:  {((usable_comments['is_pos']==0) & (usable_comments['is_neg']==0)).mean()*100:.1f}%")

Mean Sentiment Polarity: +0.3012
Positive Comments Share: 58.4%
Negative Comments Share: 11.5%
Neutral Comments Share:  30.1%


## Step 4: Topic Modeling (Train-Only Model Fitting)
To prevent future leakage, Latent Dirichlet Allocation (LDA) and vectorizers are fitted strictly on comments prior to the cutoff date (`2026-08-12`).

In [5]:
train_mask = usable_comments['date_ist'] < config.METHOD2_SPLIT_DATE
train_texts = usable_comments.loc[train_mask, 'text_clean']
all_texts = usable_comments['text_clean']

topic_dist, cv, lda = m2.fit_and_extract_lda_topics(train_texts, all_texts, n_topics=5)
vocab = cv.get_feature_names_out()

print(f"LDA Model fitted on {train_mask.sum():,} training comments (Vocab: {len(vocab)} terms)")
for k in range(5):
    top_words = [vocab[idx] for idx in lda.components_[k].argsort()[:-8:-1]]
    print(f"Topic {k}: {', '.join(top_words)}")

LDA Model fitted on 15,404 training comments (Vocab: 500 terms)
Topic 0: pro, iphone, 17, 15, vivo, max, oppo
Topic 1: phone, video, mobile, raha, hu, na, sir
Topic 2: phone, good, love, just, pixel, battery, use
Topic 3: samsung, ultra, price, nothing, phone, s25, best
Topic 4: bro, like, review, video, just, apple, anna


## Step 5: Daily Signal Aggregation and Neutral Filling
Group by `(product_id, date)` to build daily sentiment & topic signals. Missing days are filled with neutral 0.0 with `sentiment_observed == 0`.

In [6]:
daily_signals, meta = m2.build_daily_sentiment_and_topics(
    df_corpus, df_master, split_date=config.METHOD2_SPLIT_DATE, n_topics=5
)
print(f"Daily signals generated: {daily_signals.shape[0]:,} rows x {daily_signals.shape[1]} columns")
print(f"Observed comment days:   {daily_signals['sentiment_observed'].sum():,} ({daily_signals['sentiment_observed'].mean()*100:.1f}%)")
print(f"Neutral filled days:     {(daily_signals['sentiment_observed'] == 0).sum():,}")

Daily signals generated: 16,140 rows x 29 columns
Observed comment days:   4,277 (26.5%)
Neutral filled days:     11,863


## Step 6: Leakage-Free Lag & Rolling Feature Engineering
Constructs $t-1, t-2, t-3, t-7$ lags and 7-day rolling statistics using strict backward shifts (`s.shift(1).rolling(7)`). Burn-in rows (initial 7 days) are dropped.

In [7]:
df_eng, m1_feats, m2_feats = m2.engineer_method2_features(
    daily_signals, lags=[1, 2, 3, 7], rolling_window=7
)
print(f"Method 1 Numerical Features: {len(m1_feats)}")
print(f"Method 2 Enriched Features:  {len(m2_feats)}")
print(f"Valid Modeling Rows (after burn-in drop): {len(df_eng):,} (expected 15,720)")

Method 1 Numerical Features: 22
Method 2 Enriched Features:  33
Valid Modeling Rows (after burn-in drop): 15,720 (expected 15,720)


## Step 7: Chronological Train / Test Split
Enforces global cutoff at `2026-08-12`. Continuous scalers and categorical dummy encoders are fitted exclusively on training rows.

In [8]:
train_df, test_df, split_meta = m2.perform_method2_chronological_split(
    df_eng, m1_feats, m2_feats, split_date=config.METHOD2_SPLIT_DATE
)
print(f"Train Partition: {split_meta['train_window']['start_date']} to {split_meta['train_window']['end_date']} "
      f"({split_meta['train_window']['calendar_days']} days, {len(train_df):,} rows, {split_meta['train_window']['surge_events']} surges)")
print(f"Test Partition:  {split_meta['test_window']['start_date']} to {split_meta['test_window']['end_date']} "
      f"({split_meta['test_window']['calendar_days']} days, {len(test_df):,} rows, {split_meta['test_window']['surge_events']} surges)")
print(f"Total Predictor Features: {split_meta['feature_counts']['total_model_features']}")

Train Partition: 2026-01-15 to 2026-08-11 (209 days, 12,540 rows, 586 surges)
Test Partition:  2026-08-12 to 2026-10-03 (53 days, 3,180 rows, 208 surges)
Total Predictor Features: 69


## Step 8: Automated 12-Point Leakage Audit Verification
Executes the comprehensive leakage audit verifying zero lookahead bias, strict backwards windows, and quarantine isolation.

In [9]:
is_clean, audit_df = m2.run_method2_leakage_audit(
    daily_signals, df_eng, train_df, test_df, m1_feats, m2_feats,
    split_date=config.METHOD2_SPLIT_DATE, corpus_df=df_corpus
)
print(f"Audit Clean Status: {is_clean} (100% PASS/INFO, 0 FAIL)")
display_cols = ['check', 'status', 'detail']
pd.set_option('display.max_colwidth', None)
print(audit_df[display_cols].to_string(index=False))

Audit Clean Status: True (100% PASS/INFO, 0 FAIL)
                               check status                                                                                                   detail
contemporaneous_sentiment_quarantine   PASS                                Contemporaneous sentiment excluded from model features; overlapping: none
    contemporaneous_topic_quarantine   PASS                                   Contemporaneous topics excluded from model features; overlapping: none
                 lag_shift_integrity   PASS                                 Audited 107 lag index mappings for exact match Value(t-k); mismatches: 0
  rolling_trend_lookahead_quarantine   PASS        Rolling window verified as strictly backwards [t-7, t-1] using shift(1).rolling(7); violations: 0
   chronological_boundary_separation   PASS                              Max train date (2026-08-11) < Min test date (2026-08-12); zero date overlap
          nlp_topic_model_train_only   PASS             

## Step 9: Lag Cross-Correlation Diagnostics
Evaluates cross-correlations between lagged sentiment/topic indicators and continuous search interest and binary surge events.

In [10]:
lag_diag = m2.run_method2_lag_diagnostics(train_df)
print("Top Features Correlated with Demand Surges:")
print(lag_diag.head(10).to_string(index=False))

Top Features Correlated with Demand Surges:
                feature_name feature_family  corr_with_search_interest  corr_with_surge_label
          topic_camera_lag_7          topic                     0.0496                -0.0046
  topic_design_display_lag_1          topic                     0.0102                -0.0085
     topic_performance_lag_1          topic                     0.0236                -0.0097
  topic_design_display_lag_7          topic                     0.0085                -0.0109
     topic_price_value_lag_1          topic                     0.0375                -0.0120
         topic_battery_lag_1          topic                     0.0244                -0.0130
     topic_performance_lag_7          topic                     0.0253                -0.0138
           topic_lda_4_lag_1          topic                     0.0618                -0.0142
         topic_battery_lag_7          topic                     0.0205                -0.0144
sentiment_net_po

## Step 10: Visual Artifacts Inspection

In [11]:
fig_path = ROOT / 'docs' / 'figures' / 'method2_temporal_split.png'
print(f"Visual artifacts located at: {ROOT / 'docs' / 'figures'}")
for f in ['method2_temporal_split.png', 'method2_leakage_audit.png', 'method2_sentiment_topic_lags.png']:
    p = ROOT / 'docs' / 'figures' / f
    print(f"  {f:<35}: {p.stat().st_size:,} bytes")

Visual artifacts located at: /Users/mouniksai/Documents/viral-product-intelligence-/docs/figures
  method2_temporal_split.png         : 130,439 bytes
  method2_leakage_audit.png          : 182,922 bytes
  method2_sentiment_topic_lags.png   : 116,382 bytes


## Summary & Review 2 Readiness
The Method 2 data preparation pipeline has isolated sentiment and topic variables with complete leakage quarantine:
- **Zero Lookahead:** All lags and rolling trends strictly shift backwards.
- **Strict Boundary:** Train-only topic model fitting and feature scaling.
- **Multi-Modal Dataset:** `processed/method2_train.csv` (12,540 rows) and `processed/method2_test.csv` (3,180 rows) ready for benchmark modeling.